# ASG Airlines - Data Engineering and Analytics

This notebook shows the main steps used in the airline data project. The data is loaded from the Excel source, checked, cleaned, converted into analysis-ready tables and used to calculate KPIs.

**Tools used:** Python, Pandas and Power BI.

## 1. Import libraries and set the project path

The project code is kept in the `src` folder. The notebook uses the same functions from the pipeline instead of copying the complete project code into one long cell.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
        
print('Project folder:', PROJECT_ROOT)

## 2. Load the raw Excel data

The workbook contains four main sheets: flights, passengers, bookings and payments. I load the workbook once and keep the identifier fields as strings where required.

In [ ]:
from src.ingestion.excel_loader import load_raw_workbook
from src.config import RAW_EXCEL_PATH, SHEET_FLIGHTS, SHEET_PASSENGERS, SHEET_BOOKINGS, SHEET_PAYMENTS

raw_data = load_raw_workbook(PROJECT_ROOT / 'data' / 'raw' / 'UseCase - Airlines.xlsx')

flights = raw_data[SHEET_FLIGHTS]
passengers = raw_data[SHEET_PASSENGERS]
bookings = raw_data[SHEET_BOOKINGS]
payments = raw_data[SHEET_PAYMENTS]

print('Flights:', len(flights))
print('Passengers:', len(passengers))
print('Bookings:', len(bookings))
print('Payments:', len(payments))

## 3. Basic data checks

Before cleaning, I check the important fields and look for missing values and duplicate records. This helps identify problems before building the final tables.

In [ ]:
from src.validation.validators import validate_required_fields, detect_identifier_collisions

checks = {
    'Flights missing required fields': len(validate_required_fields(flights, ['flight_id', 'source', 'destination'])),
    'Passengers missing required fields': len(validate_required_fields(passengers, ['passenger_id', 'aadhaar_id'])),
    'Bookings missing required fields': len(validate_required_fields(bookings, ['booking_id', 'flight_id', 'passenger_id'])),
    'Payments missing required fields': len(validate_required_fields(payments, ['payment_id', 'booking_id']))
}

pd.Series(checks, name='Issue Count').to_frame()

## 4. Clean and standardise the data

The cleaning step removes exact duplicates, standardises text and booking status values, handles timestamp problems and prepares the records for modelling. Passenger identifiers are masked as part of the cleaning process.

In [ ]:
from src.cleaning.clean_flights import clean_flights_data
from src.cleaning.clean_passengers import clean_passengers_data
from src.cleaning.clean_bookings import clean_bookings_data
from src.cleaning.clean_payments import clean_payments_data

flights_clean, flight_anomalies = clean_flights_data(flights)
passengers_clean, passenger_anomalies = clean_passengers_data(passengers)

colliding_passenger_ids = set(passengers_clean[passengers_clean['is_collision']]['passenger_id'].unique())
multi_leg_flight_ids = set(flights_clean[flights_clean['is_multi_leg_collision']]['flight_id'].unique())
valid_passenger_ids = set(passengers_clean['passenger_id'].unique())
valid_flight_ids = set(flights_clean['flight_id'].unique())

bookings_clean, booking_anomalies = clean_bookings_data(
    bookings, colliding_passenger_ids, multi_leg_flight_ids,
    valid_passenger_ids, valid_flight_ids
)
payments_clean, payment_anomalies = clean_payments_data(payments, bookings_clean)

print('Clean flights:', len(flights_clean))
print('Clean passengers:', len(passengers_clean))
print('Clean bookings:', len(bookings_clean))
print('Clean payments:', len(payments_clean))

## 5. Flight duration and overnight flights

Flight duration is calculated from the departure and arrival timestamps. When arrival happens on the next calendar day, the flight is marked as overnight. Very unusual durations are flagged as outliers instead of being silently removed.

In [ ]:
duration_view = flights_clean[[
    'flight_id', 'departure_time', 'arrival_time',
    'duration_minutes', 'is_overnight', 'is_duration_outlier'
]].head(10)
duration_view

## 6. Build the Gold data model

The cleaned data is converted into a simple star-schema model. This creates fact tables for flights and bookings and dimension tables for airlines, routes, flights and passengers. Ambiguous references are kept as unresolved rather than forcing an incorrect match.

In [ ]:
from src.modeling.star_schema import build_star_schema

star_tables = build_star_schema(
    flights_clean, passengers_clean, bookings_clean, payments_clean
)

for name, df in star_tables.items():
    print(f'{name}: {len(df):,} rows')

## 7. Data quality anomalies and KPIs

The final stage records the issues found during cleaning and calculates the business KPIs used in the Power BI report.

In [ ]:
from src.analytics.anomaly_reporter import generate_anomaly_report
from src.analytics.kpis import compute_all_kpis

all_anomalies = flight_anomalies + passenger_anomalies + booking_anomalies + payment_anomalies
anomalies_df = generate_anomaly_report(
    all_anomalies,
    PROJECT_ROOT / 'data' / 'gold' / 'data_quality_anomalies_report.csv'
)

kpis = compute_all_kpis(
    star_tables['fact_flights'],
    star_tables['fact_bookings'],
    payments_clean,
    star_tables['dim_airline'],
    star_tables['dim_route'],
    anomalies_df
)

print('Total anomalies:', len(anomalies_df))
print('Flight KPI table:', kpis['flight_kpis'].shape)
print('Booking KPI table:', kpis['booking_kpis'].shape)
print('Payment KPI table:', kpis['payment_kpis'].shape)

## 8. Final output

The Gold layer contains the analysis-ready CSV files used by the Power BI report. The dashboard focuses on flight duration, route traffic, airline distribution, booking status, payments, revenue and data quality.

The complete project therefore follows this flow:

**Raw Excel → Validation → Cleaning → Gold Model → KPIs → Power BI Dashboard**